# Key Considerations for Your Dataproc Cluster

## 1. Cluster Resources

- **Master:** n2-standard-4 (4 vCPUs, 16 GB RAM, 32GB disk)
- **Workers (2x):** n2-standard-4 (4 vCPUs, 16 GB RAM, 64GB disk each)
- **Total:** 8 worker vCPUs, ~32 GB RAM (excluding master node)


## 2. Dataproc Features Disabled

- No autoscaling, Metastore, advanced execution layer, advanced optimizations
- **Storage:** pd-balanced (no SSDs, so I/O optimization is crucial)
- **Networking:** Internal IP enabled


## 3. Optimization Strategy

- Tune shuffle partitions, broadcast join threshold, and storage persistence
- Adjust parallelism based on 2 workers × 4 cores
- Avoid excessive caching due to disk-based storage


In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName('Olist Ecommerce Performance Optimization') \
    .config('spark.executor.memory', '6g') \
    .config('spark.executor.cores', '4') \
    .config('spark.executor.instances', '2') \
    .config('spark.driver.memory', '4g') \
    .config('spark.driver.maxResultSize', '2g') \
    .config('spark.sql.shuffle.partitions', '64') \
    .config('spark.default.parallelism', '64') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartition.enabled', 'true') \
    .config('spark.sql.autoBroadcastJoinThreshold', 20*1024*1024) \
    .config('spark.sql.files.maxPartitionBytes', '64MB') \
    .config('spark.sql.files.openCostInBytes', '2MB') \
    .config('spark.memory.fraction', 0.8) \
    .config('spark.memory.storageFraction', 0.2) \
    .getOrCreate()

26/02/15 09:27:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
!hadoop fs -ls /data/olist_proc/

Found 11 items
drwxr-xr-x   - root hadoop          0 2026-02-14 16:13 /data/olist_proc/cleaned_data.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/customers_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-15 07:57 /data/olist_proc/full_orders_df_op.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:22 /data/olist_proc/geolocation_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/order_item_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/orders_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:21 /data/olist_proc/payments_df.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:37 /data/olist_proc/product_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:19 /data/olist_proc/products_df_cleaned.parquet
drwxr-xr-x   - root hadoop          0 2026-02-14 16:20 /data/olist_proc/reviews_df_cleaned.parquet
drwxr-xr

In [4]:
customers_df = spark.read.parquet('/data/olist_proc/customers_df_cleaned.parquet')
order_item_df = spark.read.parquet('/data/olist_proc/order_item_df_cleaned.parquet')
payments_df = spark.read.parquet('/data/olist_proc/payments_df.parquet')
sellers_df = spark.read.parquet('/data/olist_proc/sellers.parquet')
geolocation_df = spark.read.parquet('/data/olist_proc/geolocation_df.parquet')
products_df = spark.read.parquet('/data/olist_proc/products_df_cleaned.parquet')
reviews_df = spark.read.parquet('/data/olist_proc/reviews_df_cleaned.parquet')
orders_df = spark.read.parquet('/data/olist_proc/orders_df_cleaned.parquet')

In [5]:
full_orders_df_op = spark.read.parquet('/data/olist_proc/full_orders_df_op.parquet')

In [6]:
full_orders_df_op.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (null

## Optimized Join Strategies

In [9]:
from pyspark.sql.functions import *

In [10]:
# Broadcast
customers_broadcast_df = broadcast(customers_df)
optimized_broadcast_join = full_orders_df_op.join(customers_broadcast_df, 'customer_id')

In [11]:
# Sort and Merge join

sorted_customers_df = customers_df.sortWithinPartitions('customer_id')
sorted_orders_df = full_orders_df_op.sortWithinPartitions('customer_id')

optimized_merge_full_orders_df = sorted_orders_df.join(sorted_customers_df, 'customer_id')

In [12]:
# Bucket join

bucketed_customers_df = customers_df.repartition(10, 'customer_id')
bucketed_orders_df = full_orders_df_op.repartition(10, 'customer_id')

bucket_join_df = bucketed_orders_df.join(bucketed_customers_df, 'customer_id')

In [13]:
spark.stop()

In [2]:
spark.stop()